<h4> RAG Implementation in Python on HR - policy Document </h4>
<UL>
<li>Read the company HR policy PDF using pypdf and chunk it</li>
<li> Vectorize the PDF </li>
<li> Store the vectorized PDF in Pinecone </li>
<li> Semantic Search - vertorize the user_query embedding and search it in pinecone and get the top results </li>
<li> Send to Groq LLM , the user query and the results of the query for the output </li>
</ul>


In [1]:
# pip install pypdf pinecone 
!pip install -r requirements.txt

In [2]:
from pypdf import PdfReader
import os
pdf_path = r".\resources\HR_Policy_Sample.pdf"
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found at {pdf_path}")
reader = PdfReader(pdf_path)
pages = [page.extract_text() for page in reader.pages]
print(f"Extracted text from {len(pages)} pages of the PDF.")
print(pages[0])  # Print the text of the first page for verification

Extracted text from 4 pages of the PDF.
HUMAN RESOURCES POLICY
 HANDBOOK
Effective Date: January 1, 2024
This handbook outlines the policies and procedures of our organization to promote a positive,
professional, and productive workplace environment. All employees are expected to review
and comply with these policies. For questions regarding any policy, please contact the Human
Resources Department.
Table of Contents
1. Introduction and Equal Opportunity
Page 1
2. Code of Conduct and Professional Behavior
Page 2
3. Attendance and Time Off Policies
Page 2
4. Anti-Harassment and Discrimination
Page 3
5. Workplace Safety and Health
Page 4



In [3]:
#extract  900 characters from the PDF pages  and store in a list
# Add overlap of 150 characters between chunks incase if the sentence is cut off in the middle of a chunk
from traitlets import List


def chunk_pages(pages: List[str], chunk_size: int = 900, chunk_overlap: int = 150) -> List[str]:
    chunks: List[str] = []
    full_text = " ".join(pages)
    text_length = len(full_text)
    if text_length == 0:
        return chunks

    start = 0
    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = full_text[start:end].strip()
        if chunk:  # Only add non-empty chunks
            chunks.append(chunk)
        if end >= text_length:
            break
        start = end - chunk_overlap
        print(f"Starting new chunk from index {start}")
    return chunks





In [4]:
chunks = chunk_pages(pages, chunk_size=900, chunk_overlap=150)
print(f"Created {len(chunks)} chunks from the PDF text.")
print(chunks[0])  # Print the first chunk for verification

Starting new chunk from index 750
Starting new chunk from index 1500
Starting new chunk from index 2250
Starting new chunk from index 3000
Starting new chunk from index 3750
Starting new chunk from index 4500
Starting new chunk from index 5250
Created 8 chunks from the PDF text.
HUMAN RESOURCES POLICY
 HANDBOOK
Effective Date: January 1, 2024
This handbook outlines the policies and procedures of our organization to promote a positive,
professional, and productive workplace environment. All employees are expected to review
and comply with these policies. For questions regarding any policy, please contact the Human
Resources Department.
Table of Contents
1. Introduction and Equal Opportunity
Page 1
2. Code of Conduct and Professional Behavior
Page 2
3. Attendance and Time Off Policies
Page 2
4. Anti-Harassment and Discrimination
Page 3
5. Workplace Safety and Health
Page 4
 1. Code of Conduct and Professional Behavior
Professional Standards
All employees are expected to maintain the high

In [5]:
#pip install sentence-transformers scikit-learn openai
!pip install -r requirements.txt

<h4> Create the Vector Embeddings of the HR Policy PDF document </h4>

In [6]:
from typing import List, Dict
import numpy as np

def create_embeddings(chunks: List[str], model_type: str = "huggingface", model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> List[List[float]]:
    """
    Create vector embeddings for each chunk of text.

    Args:
        chunks (List[str]): List of text chunks to embed
        model_type (str): Type of embedding model to use
                         Options: "huggingface", "openai"
        model_name (str): Specific model name/ID to use for HuggingFace
                         Examples:
                         - "sentence-transformers/all-MiniLM-L6-v2" (384-dim, fast, good for RAG)
                         - "sentence-transformers/all-mpnet-base-v2" (768-dim, more accurate)
                         - "intfloat/e5-large-v2" (1024-dim, high quality)

    Returns:
        List[List[float]]: List of embeddings, one for each chunk

    Example:
        >>> chunks = ["This is chunk 1", "This is chunk 2"]
        >>> embeddings = create_embeddings(chunks, model_type="huggingface")
        >>> len(embeddings)
        2
        >>> len(embeddings[0])  # Dimension of embedding
        384
    """

    if model_type.lower() == "huggingface":
        from sentence_transformers import SentenceTransformer

        print(f"Loading HuggingFace model: {model_name}")
        model = SentenceTransformer(model_name)

        print(f"Creating embeddings for {len(chunks)} chunks...")
        embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

        # Convert numpy array to list of lists
        embeddings_list = embeddings.tolist()

        print(f"✅ Created {len(embeddings_list)} embeddings with dimension {len(embeddings_list[0])}")
        return embeddings_list

    elif model_type.lower() == "openai":
        from openai import OpenAI

        client = OpenAI()
        embeddings = []

        print(f"Creating embeddings for {len(chunks)} chunks using OpenAI...")
        for i, chunk in enumerate(chunks):
            response = client.embeddings.create(
                model="text-embedding-3-small",
                input=chunk
            )
            embeddings.append(response.data[0].embedding)
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(chunks)} chunks")

        print(f"✅ Created {len(embeddings)} embeddings with dimension {len(embeddings[0])}")
        return embeddings

    else:
        raise ValueError(f"Unsupported model_type: {model_type}. Choose from 'huggingface', 'openai'")


def create_embeddings_with_metadata(chunks: List[str], model_type: str = "huggingface", model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> List[Dict]:
    """
    Create vector embeddings for each chunk and return with metadata.

    Args:
        chunks (List[str]): List of text chunks to embed
        model_type (str): Type of embedding model
        model_name (str): Specific model name

    Returns:
        List[Dict]: List of dictionaries containing chunk info and embeddings

    Example:
        >>> chunks = ["Chunk 1", "Chunk 2"]
        >>> results = create_embeddings_with_metadata(chunks)
        >>> results[0].keys()
        dict_keys(['chunk_id', 'text', 'embedding', 'text_length', 'model'])
    """
    embeddings = create_embeddings(chunks, model_type, model_name)

    embeddings_with_metadata = []
    for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        embeddings_with_metadata.append({
            "id": f"chunk_{idx}",
            "values": embedding,
            "metadata": {
                "chunk_index": idx,
                "text": chunk,
                "text_length": len(chunk),
                "model": model_name
            }
        })

    print(f"✅ Created {len(embeddings_with_metadata)} embeddings with metadata")
    return embeddings_with_metadata

In [7]:
# Test the embedding functions with your chunks
# Option 1: Simple embeddings (returns list of embedding vectors)
embeddings = create_embeddings(chunks, model_type="huggingface")
print(f"\n{'='*60}")
print(f"EMBEDDING RESULTS")
print(f"{'='*60}")
print(f"Total embeddings: {len(embeddings)}")
print(f"Embedding dimension: {len(embeddings[0])}")
print(f"\nFirst embedding (first 10 values): {embeddings[0][:10]}")
print(f"Last embedding (first 10 values): {embeddings[-1][:10]}")

c:\Sree\ForwardDeployedEngineering-Course-IK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading HuggingFace model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5987.46it/s]


Creating embeddings for 8 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.98it/s]

✅ Created 8 embeddings with dimension 384

EMBEDDING RESULTS
Total embeddings: 8
Embedding dimension: 384

First embedding (first 10 values): [-0.03193693608045578, 0.08594144135713577, -0.028329381719231606, 0.0021036204416304827, -0.015720700845122337, 0.03761535882949829, -0.014042871072888374, -0.06557917594909668, -0.08200597018003464, 0.022822901606559753]
Last embedding (first 10 values): [-0.08084948360919952, 0.13133670389652252, 0.0029888730496168137, -0.06944375485181808, 0.05405395105481148, 0.09888837486505508, 0.004703547339886427, 0.005538854748010635, -0.05181722715497017, 0.00010606784780975431]


In [8]:
# Option 2: Embeddings with metadata (useful for storing in vector DB)
embeddings_with_meta = create_embeddings_with_metadata(chunks, model_type="huggingface")

print(f"\n{'='*60}")
print(f"EMBEDDINGS WITH METADATA")
print(f"{'='*60}")
print(f"Total records: {len(embeddings_with_meta)}")
print(f"\nFirst record keys: {list(embeddings_with_meta[0].keys())}")
print(f"\nFirst record:")
print(f"  - Chunk ID: {embeddings_with_meta[0]['id']}")
print(f"  - Text length: {embeddings_with_meta[0]['metadata']['text_length']} characters")
print(f"  - Embedding dimension: {len(embeddings_with_meta[0]['values'])}")
print(f"  - Model: {embeddings_with_meta[0]['metadata']['model']}")
print(f"  - Text preview: {embeddings_with_meta[0]['metadata']['text'][:100]}...")

Loading HuggingFace model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12345.71it/s]


Creating embeddings for 8 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.57it/s]

✅ Created 8 embeddings with dimension 384
✅ Created 8 embeddings with metadata

EMBEDDINGS WITH METADATA
Total records: 8

First record keys: ['id', 'values', 'metadata']

First record:
  - Chunk ID: chunk_0
  - Text length: 899 characters
  - Embedding dimension: 384
  - Model: sentence-transformers/all-MiniLM-L6-v2
  - Text preview: HUMAN RESOURCES POLICY
 HANDBOOK
Effective Date: January 1, 2024
This handbook outlines the policies...


<h4> Store the embeddings with metadata in Pinecone DB </h4>

In [9]:
from pinecone import Pinecone
from dotenv import load_dotenv
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pincone_client = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pincone_client.Index(os.getenv("PINECONE_INDEX_NAME"))





In [10]:
# Upload to Pinecone
index.upsert(vectors=embeddings_with_meta, namespace="")

UpsertResponse(upserted_count=8)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query: str, chunks: List[str], embeddings: List[List[float]], top_k: int = 3, model_type: str = "huggingface", model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> List[Dict]:
    """
    Perform semantic search to find the most relevant chunks for a query.

    Args:
        query (str): The search query
        chunks (List[str]): Original text chunks
        embeddings (List[List[float]]): Pre-computed embeddings for chunks
        top_k (int): Number of top results to return
        model_type (str): Type of embedding model
        model_name (str): Specific model name

    Returns:
        List[Dict]: Top k most relevant chunks with similarity scores
    """
    
    # Create embedding for the query
    query_embeddings = create_embeddings([query], model_type=model_type, model_name=model_name)
    query_embedding = query_embeddings[0]
    
    # Calculate similarity between query and all chunks
    embeddings_array = np.array(embeddings)
    query_array = np.array(query_embedding).reshape(1, -1)
    
    similarities = cosine_similarity(query_array, embeddings_array)[0]
    
    # Get top k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": idx,
            "text": chunks[idx],
            "similarity_score": float(similarities[idx]),
            "text_length": len(chunks[idx])
        })
    
    return results


In [12]:
# Test semantic search
print(f"\n{'='*60}")
print(f"SEMANTIC SEARCH TEST")
print(f"{'='*60}")

test_query = "What is the code of conduct?"
print(f"\nQuery: '{test_query}'")
print(f"\nSearching for most relevant chunks...\n")

search_results = semantic_search(test_query, chunks, embeddings, top_k=3)

for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"  - Chunk ID: {result['chunk_id']}")
    print(f"  - Similarity Score: {result['similarity_score']:.4f}")
    print(f"  - Text preview: {result['text'][:500]}...\n")


SEMANTIC SEARCH TEST

Query: 'What is the code of conduct?'

Searching for most relevant chunks...

Loading HuggingFace model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6889.94it/s]


Creating embeddings for 1 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.09it/s]

✅ Created 1 embeddings with dimension 384
Result 1:
  - Chunk ID: 1
  - Similarity Score: 0.4947
  - Text preview: conduct. This
includes treating colleagues, supervisors, and clients with respect and courtesy. Employees
should perform their duties with integrity, honesty, and dedication to their work. Any behavior
that undermines the workplace environment or company values is subject to disciplinary
action, up to and including termination of employment.
Confidentiality
Employees may have access to confidential company information, including trade secrets,
client lists, financial data, and proprietary proces...

Result 2:
  - Chunk ID: 4
  - Similarity Score: 0.4533
  - Text preview: ohibit any form of harassment or discrimination based on protected characteristics.
Protected Characteristics
The company prohibits harassment and discrimination based on race, color, religion, sex,
national origin, age, disability, sexual orientation, gender identity, veteran status, or any other
characte

<h4> Semantic Search </h4>
<ul> 
<li> Create the vector representation of user query </li>
<li> Search the vector of user query in the vector DB against the vectors of HR policy document in Pinecone and fetch the top k results.
<li> Send the output along with user query to LLM 
</ul>

In [31]:

def process_user_query(query_list:List[str]):
    query_vector_result = create_embeddings(query_list)
    results = index.query(
        vector = query_vector_result,
        top_k = 4,
        include_metadata=True,
        namespace=""
    )
    matched_chunks = []
    for match in results.matches:
        matched_chunks.append(match.metadata.get('text', ''))
    return matched_chunks
    

In [32]:
user_query = "What is the work time policy?"
user_query_list = []
user_query_list.append(user_query)
search_results = process_user_query(user_query_list)
print(search_results)

Loading HuggingFace model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5011.64it/s]


Creating embeddings for 1 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.11it/s]


✅ Created 1 embeddings with dimension 384
['appropriate to their position and work\nenvironment. Business professional attire is required for client-facing roles. The company\nreserves the right to determine what constitutes appropriate workplace dress.\n2. Attendance and Time Off Policies\nWork Schedule\nStandard full-time work hours are Monday through Friday, 9:00 AM to 5:00 PM. Employees\nare expected to arrive on time and work their full schedule. Consistent tardiness or absence\nwithout proper notification may result in disciplinary action.\nPaid Time Off (PTO)\nFull-time employees receive 20 days of paid time off annually, which includes vacation days\nand sick leave. PTO requests should be submitted to your supervisor at least two weeks in\nadvance when possible. The company maintains the right to deny requests during critical\nbusiness periods. Unused PTO may be carried over to the following year with approval, up to\na maximum of 5 d', "he right to deny requests during critica

<h4> Call the LLM to get the augmented response </h4>

In [34]:
# call the groq api using langchain_groq
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
LLAMA_MODEL = os.getenv("LLAMA_MODEL")
llm = ChatGroq(api_key=groq_api_key, model=LLAMA_MODEL)
search_results_str = ''.join(search_results)
prompt = f"""You are a helpful chat assistant who answers user's questions. Answer the questions from the given context.\
Don't make any assumptions and don't answer which is not given in the context.\
User Question: {user_query} The given context is : {search_results_str}
"""

In [35]:
response = llm.invoke(prompt)
print(response.content)

The work‑time policy states that standard full‑time hours are **Monday through Friday, 9:00 AM – 5:00 PM**. Employees are expected to arrive on time and work their full scheduled day; repeated tardiness or un‑notified absences can lead to disciplinary action.
